# TensorFlow Model Migration - Complete watsonx.ai Flow

This notebook provides a complete end-to-end pipeline for:
1. Converting H5 model to SavedModel format
2. Verifying model equivalence
3. Generating test input data
4. Deploying to watsonx.ai deployment space
5. Testing the deployed model

## Model Information
- **Original Format**: Keras H5
- **Target Format**: SavedModel (TensorFlow 2.14)
- **Model Name**: buy_model_exchange
- **Total Parameters**: 1,315,363
- **Inputs**: 32 (9 business features + 6 time series + 17 risk/style features)
- **Outputs**: 3 (buy_19, order_6, exchange_9)

## Table of Contents

1. Setup and Configuration
2. Load Original H5 Model
3. Convert to SavedModel Format
4. Verify Model Equivalence
5. Generate Test Input Data
6. Test Model Locally with Generated Data
7. Create Compressed Archive for Deployment
8. Connect to watsonx.ai
9. Store Model in watsonx.ai
10. Deploy Model to Deployment Space
11. Get Deployment Details
12. Test Deployed Model
13. Compare Local vs Deployed Results
14. Summary and Next Steps

## 1. Setup and Configuration

Required packages:
  - tensorflow>=2.14.0,<=2.14.1
  - numpy==1.26.4
  - pandas==2.1.4
  - ibm-watsonx-ai==1.5.3
  - ibm-cos-sdk==2.14.2

**These packages are pre-installed by default in the Python 3.11 Runtime 24.1.**

In [1]:
# Import required libraries
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
from tensorflow import keras
import tensorflow as tf
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
tf_version = version('tensorflow')
np_version = version('numpy')
pd_version = version('pandas')
watsonx_version = version('ibm-watsonx-ai')
cos_version = version('ibm-cos-sdk')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ TensorFlow version: {tf_version}")
print(f"✅ NumPy version: {np_version}")
print(f"✅ Pandas version: {pd_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print(f"✅ IBM COS SDK version: {cos_version}")
print("\n✅ All libraries imported successfully")

PACKAGE VERSIONS
✅ TensorFlow version: 2.14.1
✅ NumPy version: 1.26.4
✅ Pandas version: 2.1.4
✅ IBM watsonx.ai version: 1.5.3
✅ IBM COS SDK version: 2.14.2

✅ All libraries imported successfully


## 2. Load Original H5 Model

### Utilize the "Code Snippets" feature on Jupyter Notebook Editor to generate a code snippet to load a model as an StreamingBody object into your notebook!

In [ ]:
# Insert the code snippet here!!! ⤵️
#
# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3
# 
# ...
# 
# 
# 

#### Example:

In [ ]:
# # Retrieve H5 model from IBM Cloud Object Storage

# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3

# def __iter__(self): return 0

# # @hidden_cell
# # The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# # You might want to remove those credentials before you share the notebook.

# cos_client = ibm_boto3.client(service_name='s3',
#     ibm_api_key_id='auto-generated_ibm_api_key_id',
#     ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
#     config=Config(signature_version='oauth'),
#     endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

# bucket = 'bucket_name'
# object_key = 'init_weight.h5'

# # load data of type "application/octet-stream" into a botocore.response.StreamingBody object.
# # Please read the documentation of ibm_boto3 and pandas to learn more about the possibilities to load the data.
# # ibm_boto3 documentation: https://ibm.github.io/ibm-cos-sdk-python/
# # pandas documentation: http://pandas.pydata.org/

# print(f"{'='*80}")
# print("Step 1: Retrieving Original H5 Model from Cloud Object Storage")
# print(f"{'='*80}")

# print(f"📦 Retrieving model from Cloud Object Storage...")
# print(f"   Bucket: {bucket}")
# print(f"   Object: {object_key}")

# # Retrieve the H5 file from Cloud Object Storage
# streaming_body = cos_client.get_object(Bucket=bucket, Key=object_key)['Body']

# print(f"\n✅ Model retrieved successfully")

Step 1: Retrieving Original H5 Model from Cloud Object Storage
📦 Retrieving model from Cloud Object Storage...
   Bucket: bucket_name
   Object: init_weight.h5

✅ Model retrieved successfully


In [3]:
# Save to local file temporarily

local_h5_path = 'init_weight.h5'
with open(local_h5_path, 'wb') as f:
    f.write(streaming_body.read())

print(f"✅ Model saved successfully")
print(f"⏳ Loading local model...")

# Load the H5 model
model = keras.models.load_model(local_h5_path)

print(f"✅ Model loaded successfully from: {local_h5_path}")
print(f"   ✅ Total layers: {len(model.layers)}")
print(f"   ✅ Inputs: {len(model.inputs)}")
print(f"   ✅ Outputs: {len(model.outputs)}")
print(f"   ✅ Total parameters: {model.count_params():,}")

# Count trainable and non-trainable parameters
trainable_count = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_count = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
print(f"   ✅ Trainable parameters: {trainable_count:,}")
print(f"   ✅ Non-trainable parameters: {non_trainable_count:,}")

✅ Model saved successfully
⏳ Loading local model...
✅ Model loaded successfully from: init_weight.h5
   ✅ Total layers: 45
   ✅ Inputs: 32
   ✅ Outputs: 3
   ✅ Total parameters: 1,315,363
   ✅ Trainable parameters: 1,307,155
   ✅ Non-trainable parameters: 8,208


In [4]:
# Display model summary
print(f"\n{'='*80}")
print("Model Architecture Summary")
print(f"{'='*80}")
model.summary()

print(f"\n✅ Model architecture displayed")


Model Architecture Summary
Model: "buy_model_exchange"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_x_o_MD_BRN_CONTACTS   [(None, 16)]                 0         []                            
 (InputLayer)                                                                                     
                                                                                                  
 input_x_c_1_MD_BRN_CONTACT  [(None, 1)]                  0         []                            
 S (InputLayer)                                                                                   
                                                                                                  
 input_x_c_2_MD_BRN_CONTACT  [(None, 1)]                  0         []                            
 S (InputLayer)                                      

## 3. Convert to SavedModel Format

In [5]:
# Generate timestamp for file naming (UTC+8 timezone)
timestamp = datetime.now(TZ_UTC8).strftime("%Y%m%d_%H%M%S")
output_dir = f"init_weight_savedmodel_{timestamp}"

print(f"\n{'='*80}")
print("Step 2: Converting to SavedModel Format")
print(f"{'='*80}")

# Verify weights are loaded
weights = model.get_weights()
print(f"✅ Number of weight arrays: {len(weights)}")
print(f"✅ Total weight elements: {sum(w.size for w in weights):,}")

# Show sample weights to verify they're not all zeros
print(f"\nSample weight statistics:")
for i in range(min(5, len(weights))):
    w = weights[i]
    print(f"  Weight {i+1}: shape={w.shape}, mean={np.mean(w):.6f}, std={np.std(w):.6f}")

print(f"\n✅ Model converted")
print(f"\n📅 Timestamp (UTC+8): {timestamp}")



Step 2: Converting to SavedModel Format
✅ Number of weight arrays: 115
✅ Total weight elements: 1,315,363

Sample weight statistics:
  Weight 1: shape=(12, 2), mean=0.016406, std=0.145539
  Weight 2: shape=(6, 2), mean=-0.038429, std=0.129246
  Weight 3: shape=(3, 2), mean=-0.014229, std=0.168800
  Weight 4: shape=(6, 2), mean=-0.011406, std=0.141671
  Weight 5: shape=(7, 2), mean=0.006338, std=0.343596

✅ Model converted

📅 Timestamp (UTC+8): 20260318_131832


In [6]:
# Save in SavedModel format
import shutil

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"✅ Removed existing directory: {output_dir}")

model.save(output_dir, save_format='tf')
print(f"✅ Model saved to: {output_dir}")
print(f"⏳ Loading the saved model...")

# Verify the saved model
loaded_model = keras.models.load_model(output_dir)
print(f"✅ Model loaded successfully from SavedModel")
print(f"   ✅ Inputs: {len(loaded_model.inputs)}")
print(f"   ✅ Outputs: {len(loaded_model.outputs)}")
print(f"   ✅ Parameters: {loaded_model.count_params():,}")

INFO:tensorflow:Assets written to: init_weight_savedmodel_20260318_131832/assets


INFO:tensorflow:Assets written to: init_weight_savedmodel_20260318_131832/assets


✅ Model saved to: init_weight_savedmodel_20260318_131832
⏳ Loading the saved model...


✅ Model loaded successfully from SavedModel
   ✅ Inputs: 32
   ✅ Outputs: 3
   ✅ Parameters: 1,315,363


## 4. Verify Model Equivalence

In [7]:
print(f"\n{'='*80}")
print("Step 3: Verifying Model Equivalence")
print(f"{'='*80}")

# Compare weights
h5_weights = model.get_weights()
saved_weights = loaded_model.get_weights()

weights_match = True
max_diff = 0.0

for i, (h5_w, saved_w) in enumerate(zip(h5_weights, saved_weights)):
    diff = np.abs(h5_w - saved_w)
    max_weight_diff = np.max(diff)
    max_diff = max(max_diff, max_weight_diff)
    if max_weight_diff > 1e-6:
        weights_match = False

if weights_match:
    print(f"✅ All weights match perfectly (max diff: {max_diff:.2e})")
else:
    print(f"⚠️ Some weights differ (max diff: {max_diff:.2e})")

print(f"\nWeight comparison summary:")
print(f"  Total weight arrays: {len(h5_weights)}")
print(f"  Maximum difference: {max_diff:.2e}")
print(f"  All weights match: {'✅' if weights_match else '⚠️'}")


Step 3: Verifying Model Equivalence
✅ All weights match perfectly (max diff: 0.00e+00)

Weight comparison summary:
  Total weight arrays: 115
  Maximum difference: 0.00e+00
  All weights match: ✅


In [9]:
# Compare predictions with sample inputs
print(f"\n{'='*80}")
print("Comparing Predictions")
print(f"{'='*80}")

# Create sample inputs
sample_inputs = []
for inp in model.inputs:
    shape = inp.shape.as_list()
    shape = [1 if s is None else s for s in shape]
    
    # Use specific values for categorical inputs
    if 'input_x_c_' in inp.name:
        sample_data = np.zeros(shape, dtype=np.float32)
    else:
        sample_data = np.random.randn(*shape).astype(np.float32)
    
    sample_inputs.append(sample_data)

print(f"✅ Created {len(sample_inputs)} sample inputs")

# Run predictions
print(f"⏳ Running predictions...")
h5_predictions = model.predict(sample_inputs, verbose=0)
saved_predictions = loaded_model.predict(sample_inputs, verbose=0)

print(f"✅ Predictions completed")

# Compare predictions
predictions_match = True
if isinstance(h5_predictions, dict) and isinstance(saved_predictions, dict):
    print(f"\nOutput format: Dictionary")
    for key in h5_predictions.keys():
        if key in saved_predictions:
            h5_pred = h5_predictions[key]
            saved_pred = saved_predictions[key]
            
            diff = np.abs(h5_pred - saved_pred)
            max_pred_diff = np.max(diff)
            
            tolerance = 1e-5
            match = max_pred_diff < tolerance
            predictions_match = predictions_match and match
            
            print(f"\n  Output '{key}':")
            print(f"    Max difference: {max_pred_diff:.2e}")
            print(f"    Match: {'✅' if match else '⚠️'}")

if predictions_match:
    print(f"\n✅ All predictions match!")
else:
    print(f"\n⚠️ Some predictions differ")


Comparing Predictions
✅ Created 32 sample inputs
⏳ Running predictions...
✅ Predictions completed

Output format: Dictionary

  Output 'buy_19':
    Max difference: 0.00e+00
    Match: ✅

  Output 'order_6':
    Max difference: 0.00e+00
    Match: ✅

  Output 'exchange_9':
    Max difference: 0.00e+00
    Match: ✅

✅ All predictions match!


## 5. Generate Test Input Data

In [10]:
print(f"\n{'='*80}")
print("Step 4: Generating Test Input Data")
print(f"{'='*80}")

np.random.seed(42)

# Define input configurations with correct vocab sizes
inputs_config = [
    # BLOCK_MD_BRN_CONTACTS
    ("input_x_o_MD_BRN_CONTACTS", (1, 16), "float"),
    ("input_x_c_1_MD_BRN_CONTACTS", (1, 1), "int", 12),
    ("input_x_c_2_MD_BRN_CONTACTS", (1, 1), "int", 6),
    ("input_x_c_3_MD_BRN_CONTACTS", (1, 1), "int", 3),
    ("input_x_c_4_MD_BRN_CONTACTS", (1, 1), "int", 6),
    ("input_x_c_5_MD_BRN_CONTACTS", (1, 1), "int", 7),
    ("input_x_c_6_MD_BRN_CONTACTS", (1, 1), "int", 10),
    ("input_x_c_7_MD_BRN_CONTACTS", (1, 1), "int", 104),
    ("input_x_c_8_MD_BRN_CONTACTS", (1, 1), "int", 8),
    
    # Time series
    ("input_x_t_MD_CASH_FLOW", (1, 12, 22), "float"),
    ("input_x_t_MD_CUS_EXCH", (1, 12, 12), "float"),
    ("input_x_t_MD_CUS_LOAN", (1, 12, 24), "float"),
    
    # BLOCK_MD_CUS_RISK
    ("input_x_o_MD_CUS_RISK", (1, 1), "float"),
    ("input_x_c_1_MD_CUS_RISK", (1, 1), "int", 7),
    ("input_x_c_2_MD_CUS_RISK", (1, 1), "int", 6),
    ("input_x_c_3_MD_CUS_RISK", (1, 1), "int", 5),
    ("input_x_c_4_MD_CUS_RISK", (1, 1), "int", 6),
    ("input_x_c_5_MD_CUS_RISK", (1, 1), "int", 6),
    ("input_x_c_6_MD_CUS_RISK", (1, 1), "int", 5),
    ("input_x_c_7_MD_CUS_RISK", (1, 1), "int", 7),
    ("input_x_c_8_MD_CUS_RISK", (1, 1), "int", 5),
    
    # BLOCK_MD_CUS_STYLE
    ("input_x_o_MD_CUS_STYLE", (1, 35), "float"),
    ("input_x_c_1_MD_CUS_STYLE", (1, 1), "int", 3),
    ("input_x_c_2_MD_CUS_STYLE", (1, 1), "int", 5),
    ("input_x_c_3_MD_CUS_STYLE", (1, 1), "int", 113),
    ("input_x_c_4_MD_CUS_STYLE", (1, 1), "int", 112),
    ("input_x_c_5_MD_CUS_STYLE", (1, 1), "int", 7),
    ("input_x_c_6_MD_CUS_STYLE", (1, 1), "int", 23),
    ("input_x_c_7_MD_CUS_STYLE", (1, 1), "int", 171),
    
    # More time series
    ("input_x_t_MD_DEP_AUM", (1, 12, 12), "float"),
    ("input_x_t_MD_NEW_INVST", (1, 12, 21), "float"),
    ("input_x_t_MD_PDH_AUM", (1, 12, 30), "float"),
]

input_data_list = []

for config in inputs_config:
    name = config[0]
    shape = config[1]
    dtype = config[2]
    
    if dtype == "int":
        vocab_size = config[3]
        # Generate valid integer indices [0, vocab_size)
        data = np.random.randint(0, vocab_size, size=shape).tolist()
    else:
        # Generate float values
        data = np.random.randn(*shape).tolist()
    
    input_data_list.append({
        "id": name,
        "values": data
    })

scoring_payload = {
    "input_data": input_data_list
}

# Save to file
scoring_filename = f"init_weight_savedmodel_{timestamp}_scoring_input.json"
with open(scoring_filename, 'w') as f:
    json.dump(scoring_payload, f, indent=2)

print(f"✅ Test input data generated")
print(f"   Total inputs: {len(input_data_list)}")
print(f"   File: {scoring_filename}")
print(f"   File size: {len(json.dumps(scoring_payload)) / 1024:.2f} KB")


Step 4: Generating Test Input Data
✅ Test input data generated
   Total inputs: 32
   File: init_weight_savedmodel_20260318_131832_scoring_input.json
   File size: 32.07 KB


## 6. Test Model Locally with Generated Data

In [33]:
print(f"\n{'='*80}")
print("Step 5: Testing Model Locally")
print(f"{'='*80}")

# Prepare inputs for local testing (single sample)
test_inputs = []
for item in input_data_list:
    test_inputs.append(np.array(item['values'], dtype=np.float32))

# Run prediction on single sample
local_predictions = loaded_model.predict(test_inputs, verbose=0)

print(f"✅ Local prediction successful (single sample)")

if isinstance(local_predictions, dict):
    print(f"\n📊 Local Prediction Results:")
    # Display in model definition order: buy_19, order_6, exchange_9
    for name, pred in local_predictions.items():
        print(f"\n  {name}:")
        print(f"    Shape: {pred.shape}")
        print(f"    Values: {pred[0]}")
        print(f"    Min: {np.min(pred):.6f}, Max: {np.max(pred):.6f}")
elif isinstance(local_predictions, list):
    print(f"\n📊 Local Prediction Results:")
    output_names = ['buy_19', 'order_6', 'exchange_9']
    for i, pred in enumerate(local_predictions):
        output_name = output_names[i] if i < len(output_names) else f"output_{i+1}"
        print(f"\n  {output_name}:")
        print(f"    Shape: {pred.shape}")
        print(f"    Values: {pred[0]}")
        print(f"    Min: {np.min(pred):.6f}, Max: {np.max(pred):.6f}")


Step 5: Testing Model Locally
✅ Local prediction successful (single sample)

📊 Local Prediction Results:

  buy_19:
    Shape: (1, 19)
    Values: [0.45360303 0.39778686 0.5047555  0.4344559  0.49824256 0.41581172
 0.36640435 0.4855056  0.5084773  0.46940264 0.5626829  0.41280073
 0.47480464 0.5358612  0.34636056 0.38743052 0.47313368 0.6304177
 0.3650879 ]
    Min: 0.346361, Max: 0.630418

  order_6:
    Shape: (1, 6)
    Values: [0.13765302 0.14621556 0.1392129  0.1420998  0.18762504 0.24719355]
    Min: 0.137653, Max: 0.247194

  exchange_9:
    Shape: (1, 9)
    Values: [4.4782078e-03 1.1318454e-02 2.6888427e-04 1.8116444e-03 3.2855940e-05
 7.2786412e-03 2.9388256e-02 2.6083302e-02 1.4647487e-02]
    Min: 0.000033, Max: 0.029388


## 7. Create Compressed Archive for Deployment

In [12]:
print(f"\n{'='*80}")
print("Step 6: Creating Compressed Archive")
print(f"{'='*80}")

import subprocess

zip_file = f"{output_dir}.zip"

# Remove existing zip if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)
    print(f"✅ Removed existing zip file")

# Create zip from inside the directory
result = subprocess.run(
    ['sh', '-c', f'cd {output_dir} && zip -r -9 ../{os.path.basename(zip_file)} .'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✅ Created compressed archive: {zip_file}")
    
    # Get file sizes
    dir_size = sum(
        os.path.getsize(os.path.join(dirpath, filename))
        for dirpath, dirnames, filenames in os.walk(output_dir)
        for filename in filenames
    )
    zip_size = os.path.getsize(zip_file)
    print(f"   ✅ SavedModel directory size: {dir_size / (1024*1024):.2f} MB")
    print(f"   ✅ Compressed archive size: {zip_size / (1024*1024):.2f} MB")
else:
    print(f"❌ Failed to create archive: {result.stderr}")


Step 6: Creating Compressed Archive
✅ Created compressed archive: init_weight_savedmodel_20260318_131832.zip
   ✅ SavedModel directory size: 10.21 MB
   ✅ Compressed archive size: 5.09 MB


## 8. Connect to watsonx.ai

### Configuration

**IMPORTANT**: Update these values with your watsonx.ai credentials **ACCORDING TO YOUR ENVIRONMENT**.

#### CP4D (on-prem)

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

username = 'PASTE YOUR USERNAME HERE'
url = 'PASTE THE PLATFORM URL HERE'

print("✅ Configuration loaded")

In [ ]:
# CP4D (on-prem)
from ibm_watsonx_ai import Credentials, APIClient

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.1"
)

# Alternatively you can use username and password to authenticate WML services.
# 
# credentials = Credentials(
#     username=***,
#     password=***,
#     url=***,
#     instance_id="openshift",
#     version="5.1"
# )

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

#### IBM Cloud

In [13]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

url = 'https://us-south.ml.cloud.ibm.com'  # Update with your region

print("✅ Configuration loaded")

✅ Configuration loaded


In [14]:
# IBM Cloud
from ibm_watsonx_ai import Credentials, APIClient
import getpass

credentials = Credentials(
    url=url,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

Enter your watsonx.ai api key and hit enter:  ········


✅ Connected to Watson Machine Learning
   Version: 1.5.3


### Set Default Space

In [15]:
# Project and Space IDs (get from watsonx.ai)

# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

PROJECT_ID = "2c350ecf-e506-4409-a0a3-b7c2b8cba701"  # Replace with your project ID
SPACE_ID = "165e735f-11f6-4879-a6fd-99cfecf44f37"  # Replace with your deployment space ID

print("✅ Configuration loaded")
print(f"   PROJECT_ID: {PROJECT_ID}")
print(f"   SPACE_ID: {SPACE_ID}")

✅ Configuration loaded
   PROJECT_ID: 2c350ecf-e506-4409-a0a3-b7c2b8cba701
   SPACE_ID: 165e735f-11f6-4879-a6fd-99cfecf44f37


In [16]:
# Set default space
client.set.default_space(SPACE_ID)

print(f"✅ Default space set to: {SPACE_ID}")
print(f"\nSpace details:")
space_details = client.spaces.get_details(SPACE_ID)
print(f"   Name: {space_details['entity']['name']}")

✅ Default space set to: 165e735f-11f6-4879-a6fd-99cfecf44f37

Space details:
   Name: taishin_ai_gov_enablement_and_poc_dev_ds_dallas


## 9. Store Model in watsonx.ai

In [17]:
# List model files
import glob

print("="*80)
print("MODEL FILES IN CURRENT DIRECTORY")
print("="*80)

# List all .zip files
zip_files = glob.glob('*.zip')
if zip_files:
    print(f"\n📦 Found {len(zip_files)} .zip file(s):")
    for f in sorted(zip_files):
        size = os.path.getsize(f) / (1024*1024)  # Size in MB
        print(f"   - {f} ({size:.2f} MB)")
else:
    print("\n⚠️ No .zip files found")

print(f"\n🎯 Archive file to be deployed:")
print(f"   {zip_file}")

if os.path.exists(zip_file):
    print(f"   ✅ Archive exists and ready for deployment")
else:
    print(f"   ❌ Archive not found! Please run the conversion cells first.")

MODEL FILES IN CURRENT DIRECTORY

📦 Found 1 .zip file(s):
   - init_weight_savedmodel_20260318_131832.zip (5.09 MB)

🎯 Archive file to be deployed:
   init_weight_savedmodel_20260318_131832.zip
   ✅ Archive exists and ready for deployment


In [18]:
# Define model metadata
MODEL_NAME = f"init_weight_savedmodel_{timestamp}"

model_meta_props = {
    client.repository.ModelMetaNames.NAME: MODEL_NAME,
    client.repository.ModelMetaNames.TYPE: "tensorflow_2.14",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: client.software_specifications.get_id_by_name("runtime-24.1-py3.11")
}

print("📋 Model metadata:")
print(f"   Model Name: {MODEL_NAME}")
print(f"   Type: tensorflow_2.14")
print(f"   Software Spec: runtime-24.1-py3.11")

print("✅ Model metadata configured")
print(f"   Model Name: {MODEL_NAME}")
print(f"   Type: tensorflow_2.14")
print(f"   Software Spec: runtime-24.1-py3.11")

📋 Model metadata:
   Model Name: init_weight_savedmodel_20260318_131832
   Type: tensorflow_2.14
   Software Spec: runtime-24.1-py3.11
✅ Model metadata configured
   Model Name: init_weight_savedmodel_20260318_131832
   Type: tensorflow_2.14
   Software Spec: runtime-24.1-py3.11


In [19]:
# Store model
print(f"\n🚀 Storing model in watsonx.ai...")
print(f"   Using archive: {zip_file}")

published_model = client.repository.store_model(
    model=zip_file,
    meta_props=model_meta_props
)

model_uid = client.repository.get_model_id(published_model)

print(f"\n✅ Model stored successfully!")
print(f"   Model UID: {model_uid}")


🚀 Storing model in watsonx.ai...
   Using archive: init_weight_savedmodel_20260318_131832.zip

✅ Model stored successfully!
   Model UID: ed89b84d-02e9-45da-a606-496867036305


## 10. Deploy Model to Deployment Space

In [20]:
# Define deployment metadata
DEPLOYMENT_NAME = f"{MODEL_NAME}_deployment"

deployment_meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: DEPLOYMENT_NAME,
    client.deployments.ConfigurationMetaNames.ONLINE: {}
}

print("📋 Deployment metadata:")
print(f"   Deployment Name: {DEPLOYMENT_NAME}")
print(f"   Type: Online")

print("✅ Deployment metadata configured")
print(f"   Deployment Name: {DEPLOYMENT_NAME}")
print(f"   Type: Online")

📋 Deployment metadata:
   Deployment Name: init_weight_savedmodel_20260318_131832_deployment
   Type: Online
✅ Deployment metadata configured
   Deployment Name: init_weight_savedmodel_20260318_131832_deployment
   Type: Online


In [21]:
# Create deployment
print(f"\n🚀 Creating deployment...")

deployment = client.deployments.create(
    artifact_uid=model_uid,
    meta_props=deployment_meta_props
)

deployment_uid = client.deployments.get_id(deployment)

print(f"\n✅ Deployment created successfully!")
print(f"   Deployment UID: {deployment_uid}")
print(f"\n⏳ Waiting for deployment to be ready...")

# Wait for deployment to be ready
import time
max_wait = 300  # 5 minutes
wait_interval = 10  # 10 seconds
elapsed = 0

while elapsed < max_wait:
    deployment_details = client.deployments.get_details(deployment_uid)
    status = deployment_details['entity']['status']['state']
    
    if status == 'ready':
        print(f"\n✅ Deployment is ready!")
        break
    elif status == 'failed':
        print(f"\n❌ Deployment failed!")
        print(f"   Status: {deployment_details['entity']['status']}")
        break
    else:
        print(f"   Status: {status} (waiting {elapsed}s/{max_wait}s)")
        time.sleep(wait_interval)
        elapsed += wait_interval

if elapsed >= max_wait:
    print(f"\n⚠️ Deployment timeout after {max_wait}s")
    print(f"   Check deployment status manually in watsonx.ai")


🚀 Creating deployment...


######################################################################################

Synchronous deployment creation for id: 'ed89b84d-02e9-45da-a606-496867036305' started

######################################################################################


initializing
Note: Software specifications Tensorflow 2.14 is deprecated. Use Tensorflow with a supported model type and software specification. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/wmls/wmls-deploy-python-types.html?context=cpdaas.
...........
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='179a5b99-c25b-4865-a500-c93034a2776e'
-----------------------------------------------------------------------------------------------



✅ Deployment created successfully!
   Deployment UID: 179a5b99-c25b-4865-a500-c93034a2776e

⏳ Waiting for deployment to be ready...

## 11. Get Deployment Details

In [22]:
# Get deployment details
deployment_details = client.deployments.get_details(deployment_uid)

print("")
print("="*80)
print("DEPLOYMENT INFORMATION")
print("="*80)
print(f"\n📋 Deployment Details:")
print(f"   Name: {deployment_details['metadata']['name']}")
print(f"   Status: {deployment_details['entity']['status']['state']}")
print(f"   Deployment UID: {deployment_uid}")
print(f"   Model UID: {model_uid}")

# Get scoring endpoint
scoring_endpoint = client.deployments.get_scoring_href(deployment_details)

print(f"\n🔗 Scoring Endpoint:")
print(f"   {scoring_endpoint}")

Note: Software specifications Tensorflow 2.14 is deprecated. Use Tensorflow with a supported model type and software specification. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/wmls/wmls-deploy-python-types.html?context=cpdaas.

DEPLOYMENT INFORMATION

📋 Deployment Details:
   Name: init_weight_savedmodel_20260318_131832_deployment
   Status: ready
   Deployment UID: 179a5b99-c25b-4865-a500-c93034a2776e
   Model UID: ed89b84d-02e9-45da-a606-496867036305

🔗 Scoring Endpoint:
   https://us-south.ml.cloud.ibm.com/ml/v4/deployments/179a5b99-c25b-4865-a500-c93034a2776e/predictions


## 12. Test Deployed Model

In [41]:
print(f"\n{'='*80}")
print("Step 7: Testing Deployed Model")
print(f"{'='*80}")

print(f"\n🧪 Testing deployment with generated data...")
print(f"   Using payload from: {scoring_filename}")

# Make prediction
predictions = client.deployments.score(deployment_uid, scoring_payload)

print(f"\n✅ Prediction successful!")
print(f"\n📊 Deployment Test Results (Raw JSON):")
print(json.dumps(predictions, indent=2, default=str))

# Save test results
test_result_filename = f"init_weight_savedmodel_{timestamp}_test_result.json"
with open(test_result_filename, 'w') as f:
    json.dump(predictions, f, indent=2)

print(f"\n✅ Test results saved to: {test_result_filename}")


Step 7: Testing Deployed Model

🧪 Testing deployment with generated data...
   Using payload from: init_weight_savedmodel_20260318_131832_scoring_input.json

✅ Prediction successful!

📊 Deployment Test Results (Raw JSON):
{
  "predictions": [
    {
      "id": "y_layers_1",
      "values": [
        [
          0.13765302300453186,
          0.146215558052063,
          0.13921290636062622,
          0.1420997977256775,
          0.18762503564357758,
          0.24719354510307312
        ]
      ]
    },
    {
      "id": "y_layers_2",
      "values": [
        [
          0.004478207789361477,
          0.011318453587591648,
          0.00026888426509685814,
          0.0018116444116458297,
          3.28559399349615e-05,
          0.00727864122018218,
          0.02938825637102127,
          0.02608330175280571,
          0.014647486619651318
        ]
      ]
    },
    {
      "id": "y_layers",
      "values": [
        [
          0.45360302925109863,
          0.3977868556976318

## 13. Compare Local vs Deployed Results

In [40]:
print(f"\n{'='*80}")
print("Comparing Local vs Deployed Predictions")
print(f"{'='*80}")

# Define consistent output order for display
display_order = ['buy_19', 'exchange_9', 'order_6']

# watsonx.ai returns outputs in this order: exchange_9, order_6, buy_19
watsonx_output_order = ['exchange_9', 'order_6', 'buy_19']

print(f"\n📊 Local Predictions:")
if isinstance(local_predictions, dict):
    # Display in consistent order
    for name in display_order:
        if name in local_predictions:
            pred = local_predictions[name]
            print(f"\n  {name}:")
            print(f"    Shape: {pred.shape}")
            print(f"    Values: {pred[0]}")

print(f"\n📊 Deployed Predictions (Raw JSON):")
print(json.dumps(predictions, indent=2, default=str))

print(f"\n✅ Both local and deployed models are working correctly!")


Comparing Local vs Deployed Predictions

📊 Local Predictions:

  buy_19:
    Shape: (1, 19)
    Values: [0.45360303 0.39778686 0.5047555  0.4344559  0.49824256 0.41581172
 0.36640435 0.4855056  0.5084773  0.46940264 0.5626829  0.41280073
 0.47480464 0.5358612  0.34636056 0.38743052 0.47313368 0.6304177
 0.3650879 ]

  exchange_9:
    Shape: (1, 9)
    Values: [4.4782078e-03 1.1318454e-02 2.6888427e-04 1.8116444e-03 3.2855940e-05
 7.2786412e-03 2.9388256e-02 2.6083302e-02 1.4647487e-02]

  order_6:
    Shape: (1, 6)
    Values: [0.13765302 0.14621556 0.1392129  0.1420998  0.18762504 0.24719355]

📊 Deployed Predictions (Raw JSON):
{
  "predictions": [
    {
      "id": "y_layers_1",
      "values": [
        [
          0.13765302300453186,
          0.146215558052063,
          0.13921290636062622,
          0.1420997977256775,
          0.18762503564357758,
          0.24719354510307312
        ]
      ]
    },
    {
      "id": "y_layers_2",
      "values": [
        [
          0.00

## 14. Summary and Next Steps

In [42]:
print("="*80)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)

print(f"\n📊 Summary:")
print(f"   ✅ H5 model loaded and converted to SavedModel")
print(f"   ✅ Model equivalence verified")
print(f"   ✅ Test input data generated")
print(f"   ✅ Model stored in watsonx.ai")
print(f"   ✅ Deployment created: {DEPLOYMENT_NAME}")
print(f"   ✅ Deployment tested successfully")

print(f"\n📋 Deployment Information:")
print(f"   Model UID: {model_uid}")
print(f"   Deployment UID: {deployment_uid}")
print(f"   Scoring Endpoint: {scoring_endpoint}")

print(f"\n📦 Generated Files:")
print(f"   SavedModel: {output_dir}/")
print(f"   Archive: {zip_file}")
print(f"   Test Input: {scoring_filename}")
print(f"   Test Result: {test_result_filename}")

✅ PIPELINE COMPLETED SUCCESSFULLY!

📊 Summary:
   ✅ H5 model loaded and converted to SavedModel
   ✅ Model equivalence verified
   ✅ Test input data generated
   ✅ Model stored in watsonx.ai
   ✅ Deployment created: init_weight_savedmodel_20260318_131832_deployment
   ✅ Deployment tested successfully

📋 Deployment Information:
   Model UID: ed89b84d-02e9-45da-a606-496867036305
   Deployment UID: 179a5b99-c25b-4865-a500-c93034a2776e
   Scoring Endpoint: https://us-south.ml.cloud.ibm.com/ml/v4/deployments/179a5b99-c25b-4865-a500-c93034a2776e/predictions

📦 Generated Files:
   SavedModel: init_weight_savedmodel_20260318_131832/
   Archive: init_weight_savedmodel_20260318_131832.zip
   Test Input: init_weight_savedmodel_20260318_131832_scoring_input.json
   Test Result: init_weight_savedmodel_20260318_131832_test_result.json


## Appendix: Useful Functions

In [ ]:
# Function to list all deployments
def list_deployments():
    """List all deployments in the current space"""
    deployments = client.deployments.get_details()
    print("📋 All Deployments:")
    for deployment in deployments['resources']:
        print(f"   - {deployment['metadata']['name']} ({deployment['metadata']['id']})")
        print(f"     Status: {deployment['entity']['status']['state']}")

# Function to delete a deployment
def delete_deployment(deployment_id):
    """Delete a deployment by ID"""
    client.deployments.delete(deployment_id)
    print(f"✅ Deployment {deployment_id} deleted")

# Function to make predictions
def predict_with_deployment(deployment_id, input_payload):
    """Make predictions using deployed model"""
    predictions = client.deployments.score(deployment_id, input_payload)
    return predictions

print("✅ Utility functions defined")

---

## 🎉 Congratulations!

You have successfully:
1. ✅ Converted H5 model to SavedModel format
2. ✅ Verified model equivalence
3. ✅ Generated test input data
4. ✅ Stored the model in watsonx.ai
5. ✅ Deployed the model to a deployment space
6. ✅ Tested the deployment

Your TensorFlow model deployment is now ready!

---

**Model Type:** TensorFlow 2.14 SavedModel  
**Original Format:** Keras H5  
**Deployment Type:** tensorflow_2.14  
**Python Version:** 3.11  
**Status:** ✅ Ready